# Week 5 · Class 2 — Inference & A/B Testing in Python 🔬📈

**90 min · hands-on** · tools: `numpy`, `pandas`, `matplotlib`, `scipy.stats`, `statsmodels`

Tuesday we went from a **model to data** (probability). Today we go from **data back to the world** (inference): how much can a sample tell us, and when is a difference real?
As on Tuesday, every idea is checked with a **simulation** first, then used on **data**.

| Part | Time | What we do |
|---|---|---|
| 0 | 3' | Setup |
| 1 | 10' | Sample vs population: why estimates vary |
| 2 | 12' | The central limit theorem |
| 3 | 8' | Standard error and the √n law |
| 4 | 12' | Confidence intervals |
| 5 | 15' | Hypothesis tests: one-sample, two groups, paired |
| 6 | 10' | Errors and power |
| 7 | 15' | A/B testing end to end |
| 8 | 5' | Final challenge + recap |

🧑‍💻 = we write it **together** · ✋ **Your turn** = 2–3 minutes on your own, then we compare.

### 📂 Datasets used in this notebook
| File | What it is | Used in |
|---|---|---|
| `wearables_daily.csv` | same file as Tuesday and Week 3: 250 users × 365 days = **91,250 user-days** (steps, sleep, resting heart rate, device, age group…) | Parts 1–6 |
| `blood_pressure.csv` | **100 patients**, blood pressure `before` and `after` a treatment | Part 5 (paired test) |
| `ab_test.csv` | **50 users per version**, minutes spent on a page with version `a` and version `b` | Part 7 (A/B test on means) |
| *(simulated)* | a conversion A/B test with 10,000 users per group, generated with numpy | Parts 6–7 |

📁 **Before running:** put the three CSV files next to this notebook.
- **Colab:** click the folder icon on the left and drag the files in (they are deleted when the session ends, so upload them again next time). `statsmodels` is already installed in Colab.
- **VS Code / Jupyter:** keep the files in the same folder as the notebook, and install statsmodels once with `uv add statsmodels`.

## 0 · Setup ⚙️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

rng = np.random.default_rng(42)

In [ ]:
df = pd.read_csv("wearables_daily.csv", parse_dates=["date"])
data = df[df["steps"] < 60_000].dropna(subset=["sleep_hours"]).copy()   # same cleaning as Tuesday

bp = pd.read_csv("blood_pressure.csv")
ab = pd.read_csv("ab_test.csv")

print("wearables:", data.shape, "| blood pressure:", bp.shape, "| ab_test:", ab.shape)

---
## 1 · Sample vs population: why estimates vary 🎯 (10')

In real life we never see the whole population. Here we **cheat on purpose**: we treat all 88,000 user-days as the **population**, so we know the true answer and can check how good a small sample is.

| | Population (unknown in real life) | Sample (what we see) |
|---|---|---|
| mean | μ | x̄ |
| std | σ | s |

In [ ]:
# ✍️ live coding


🧑‍💻 **Code with me:** take 5 different random samples of 25 user-days. Do we get the same mean?

In [ ]:
# ✍️ live coding


Every sample gives a **different** x̄. That's not a mistake: it's **sampling variability**. Inference is about measuring it.

### What if the sample is biased?
Suppose we only survey users aged 20–29 (it was easier to reach them). Even a **huge** sample won't fix that.

In [ ]:
# ✍️ live coding


> ⚠️ A big biased sample gives a **precise wrong answer**. Random sampling matters more than sample size.

---
## 2 · The central limit theorem 🔔 (12')

`steps` is clearly **not** Normal: most days are moderate and a few are very active (right-skewed).

In [ ]:
# ✍️ live coding


🧑‍💻 **Code with me:** draw 10,000 samples of size n, compute each mean, and look at the histogram of the means for n = 2, 10 and 50.

In [ ]:
# ✍️ live coding


With n = 2 the means are still skewed; with n = 50 they look Normal, and the red curve predicted by the CLT fits.

$$\bar{x} \approx \mathrm{Normal}\left(\mu,\ \frac{\sigma}{\sqrt{n}}\right) \quad \text{for } n \text{ large enough}$$

✋ **Your turn (2'):** use a **QQ plot** (Week 3!) to compare the raw steps and the means of n = 50 with a Normal.

In [ ]:
# ✍️ live coding


---
## 3 · Standard error and the √n law 📏 (8')

The **standard error** is the std of the sampling distribution: how far x̄ typically lands from μ.

$$\mathrm{SE} = \frac{\sigma}{\sqrt{n}} \approx \frac{s}{\sqrt{n}}$$

In [ ]:
# ✍️ live coding


📌 Each time n is multiplied by **4**, the SE is **halved**. Precision gets expensive fast.

In practice we have **one** sample and estimate the SE from it:

In [ ]:
# ✍️ live coding


> 🧠 **s vs SE:** `s` describes how different **individuals** are (≈ 5 bpm). `SE` describes how precise our **estimate** is (≈ 0.5 bpm with n = 100).

---
## 4 · Confidence intervals 📐 (12')

$$\text{95\% CI} = \bar{x} \pm t^{*}\cdot\frac{s}{\sqrt{n}} \qquad (t^{*} \approx 1.96 \text{ for large } n)$$

🧑‍💻 **Code with me:** a 95% CI for the mean resting heart rate from a sample of 50 user-days. By hand, then with scipy.

In [ ]:
# ✍️ live coding


### What does 95% mean? Build 100 intervals and count how many contain the true μ.

In [ ]:
# ✍️ live coding


📌 The **method** is right about 95% of the time. Any single interval either contains μ or not; the red ones are unlucky samples.

### CI for a proportion: share of days hitting the 10,000-step goal
$$\hat{p} \pm 1.96\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

In [ ]:
# ✍️ live coding


> 🎲 If your interval misses the true p, you just met one of the 5%! Run the cell again a few times: about 19 out of 20 intervals contain it.

✋ **Your turn (3'):** how does the width of the 95% CI for the mean resting HR change with n? Compute it for n = 25, 100, 400, 1600. What pattern do you see?

In [ ]:
# ✍️ live coding


---
## 5 · Hypothesis tests ⚖️ (15')

The recipe: **(1)** state H₀ and H₁ · **(2)** pick α = 0.05 · **(3)** compute a test statistic · **(4)** p-value → decide.

### 5.1 One-sample t-test: is the average still 8,000 steps?
The product team claims a new onboarding changed daily steps, compared with the historical 8,000. We look at a random sample of 400 user-days.
H₀: μ = 8,000 · H₁: μ ≠ 8,000.

In [ ]:
# ✍️ live coding


### What is a p-value, really? Simulate a world where H₀ is true
We shift the population so its mean is **exactly 8,000** (H₀ true), draw 10,000 samples of 400, and count how often the mean is at least as far from 8,000 as ours.

In [ ]:
# ✍️ live coding


📌 The p-value = how often **chance alone** (H₀ true) produces a result at least as extreme as ours. It is **not** the probability that H₀ is true.

✋ **Your turn (2'):** the true mean of our population is about 8,370 steps, so here H₀ is actually false. Repeat the test with samples of **n = 50** and **n = 2,000**. What happens to the p-value, and why?

In [ ]:
# ✍️ live coding


### 5.2 Two independent groups: do Apple Watch users sleep more than Fitbit users?
We use **Welch's t-test** (`equal_var=False`): the safe default when the two groups may have different spreads.

In [ ]:
# ✍️ live coding


🧑‍💻 **Code with me:** now use **all** the rows (more than 13,000 per group). What changes?

In [ ]:
# ✍️ live coding


📌 With huge n even small differences become "significant". The question changes from *"is it real?"* to *"is it big enough to matter?"* → always report the **effect size**.

### 5.3 Paired t-test: did the treatment lower blood pressure?
The **same 100 patients** were measured before and after. Each patient is compared with **themselves**, which removes the differences between people.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


✋ **Your turn (2'):** which test would you use for these questions?
1. Is the average sleep of Garmin users different from 7 hours?
2. Do users walk more on Saturdays than on Mondays? (the same users, measured on both days)
3. Is hitting the step goal related to the age group?

*Answers:* 1) one-sample t-test · 2) paired t-test · 3) χ² test (`stats.chi2_contingency`, Week 3).

---
## 6 · Errors and power 🎲 (10')

### 6.1 False alarms: run 2,000 experiments where H₀ is TRUE
Both groups come from the **same** population (sleep hours). Any "significant" result is a **Type I error**.

In [ ]:
# ✍️ live coding


### 6.2 Power: a feature REALLY adds 10 minutes of sleep. How often do we detect it?
🧑‍💻 **Code with me:** simulate the experiment for different group sizes n and count how often p < 0.05.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


📌 With 50 users per group, a real 10-minute effect is missed most of the time. **Plan n before** the experiment.

---
## 7 · A/B testing end to end 🧪 (15')

### 7.1 Conversion test: does the new "Start workout" button work?
We simulate the test from the slides: 10,000 users per group, true conversion 10% (A) and 11% (B).

In [ ]:
# ✍️ live coding


🧑‍💻 **Code with me:** z-test for two proportions + 95% CI of the difference.

In [ ]:
# ✍️ live coding


### 7.2 Sample size *before* the test
How many users per group to detect 10% → 11% with 80% power?

In [ ]:
# ✍️ live coding


### 7.3 ⚠️ The peeking problem
An **A/A test**: both groups are identical (10% each), so there is **nothing** to find. The analyst checks the p-value **every day for 20 days** (500 new users per group per day) and stops at the first p < 0.05.

In [ ]:
# ✍️ live coding


📌 Peeking every day multiplies the false-positive rate. Fix the sample size first and analyse **once**.

### 7.4 A/B test on a mean: minutes spent on the page (`ab_test.csv`)
50 users saw version `a`, 50 users saw version `b`. Did version b keep users longer?

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


✋ **Your turn (2'):** write one sentence for the product manager: what is the effect, how sure are we, and what would you recommend?

---
## 8 · Final challenge 🏁 (5')

The exercise from the slides: 800 users, B gets a sleep-coaching feature.
A: n = 400, mean 7.10 h, s = 0.8 · B: n = 400, mean 7.22 h, s = 0.8.

✋ **Your turn:** compute the test and the 95% CI of the difference with scipy, using only these summary numbers (hint: `stats.ttest_ind_from_stats`). Then translate the effect into minutes.

In [ ]:
# ✍️ live coding


*Answer:* significant at 5% (p ≈ 0.03), about **+7 minutes** of sleep, but plausibly anything between ~0.5 and ~14 minutes. Significant, small and uncertain → report the CI and let the business decide.

## 🧭 Recap

| Idea | Python |
|---|---|
| standard error | `stats.sem(x)` = `x.std(ddof=1) / np.sqrt(len(x))` |
| 95% CI for a mean | `stats.t.interval(0.95, df=n-1, loc=x.mean(), scale=stats.sem(x))` |
| one mean vs a value | `stats.ttest_1samp(x, value)` |
| two independent groups | `stats.ttest_ind(a, b, equal_var=False)` (Welch) |
| same units before / after | `stats.ttest_rel(after, before)` |
| two proportions | `proportions_ztest(counts, nobs)` + `confint_proportions_2indep(...)` |
| sample size / power | `NormalIndPower().solve_power(effect_size=..., alpha=0.05, power=0.8)` |

**Rules to remember:** random samples beat big samples · report effect size + CI, not only p · plan n before the test · never stop at the first p < 0.05.

🧪 **Saturday lab:** hypothesis testing (including `blood_pressure.csv`).